In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Load your dataset (replace 'your_dataset.csv' with your actual file)
data = pd.read_csv('Fertilizer_pre.csv')

# Correct feature names
data.rename(columns={'Temparature': 'Temperature', 'Humidity ': 'Humidity'}, inplace=True)

# Separate features and target variable
X = data.drop(columns=['Fertilizer Name'])
y = data['Fertilizer Name']

# Map fertilizer names to numeric labels (if not done already)
fertilizer_mapping = {
    '10-26-26': 0,
    '14-35-14': 1,
    '17-17-17': 2,
    '20-20': 3,
    '28-28': 4,
    'DAP': 5,
    'Urea': 6
}
y = y.map(fertilizer_mapping)

# Encode categorical features
le_soil = LabelEncoder()
le_crop = LabelEncoder()
X['Soil Type'] = le_soil.fit_transform(X['Soil Type'])
X['Crop Type'] = le_crop.fit_transform(X['Crop Type'])

# Split data into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Initialize and train the model
model = XGBClassifier()
model.fit(X_train, y_train)

# Make predictions on the validation set
y_val_pred = model.predict(X_val)

# Evaluate model accuracy on validation set
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"Validation accuracy: {val_accuracy:.2f}")

# Example prediction for new data
new_data = pd.DataFrame({
    'Temperature': [28],
    'Humidity': [54],
    'Moisture': [46],
    'Soil Type': [le_soil.transform(['Clayey'])[0]],
    'Crop Type': [le_crop.transform(['Wheat'])[0]],
    'Nitrogen': [30],
    'Potassium': [4],
    'Phosphorous': [3]
})

predicted_label = model.predict(new_data)[0]
predicted_fertilizer = next(key for key, value in fertilizer_mapping.items() if value == predicted_label)
print(f"Predicted fertilizer: {predicted_fertilizer}")
